# Week 1: Data acquisition and audit (Group 2) — RAM-safe Colab version

**Project:** Can We Trust AI-Image Detectors on Ugandan Content?
**Member default:** AW (Andrina Watsemba)

### IMPORTANT: Run ONE member per Colab session
Do not try to run all 6 members in one notebook session — Colab will run out of RAM and the kernel will restart.

1. Set `MEMBER` to your code below (e.g. `AW`).
2. Run cells **one at a time** — do not click "Run all".
3. If the kernel crashes on a source, restart the runtime and resume: earlier sources are already saved.
4. Keep `PILOT = True` until you confirm it works without crashing.

### Drive structure
- Root: `/content/drive/MyDrive/Deepfake Detection Project/datasets`
- Subfolders: `public_data/`, `in_the_wild/`, `metadata/`, `audit_outputs/`
- Metadata: `public_openfake_AW.csv`, `public_community_AW.csv`, `public_reddit_AW.csv`

In [ ]:
import os, sys
print('Python:', sys.version.split()[0])

# Rough RAM check
try:
    import psutil
    mem = psutil.virtual_memory()
    print(f'RAM: {mem.available / 1e9:.1f} GB free of {mem.total / 1e9:.1f} GB total')
except ImportError:
    print('psutil not installed (RAM check skipped)')

print('RAM check cell done')

In [ ]:
!pip -q install -U datasets imagehash tqdm psutil

In [ ]:
import io, os, re, time, json, glob, math, hashlib, random, gc, sys
from collections import Counter
import numpy as np, pandas as pd
from PIL import Image as PILImage

print('Imports OK')

## 1. Configuration (edit `MEMBER` and `PILOT` only)

In [ ]:
# ===========================================================================
# CURRENT DRIVE ROOT
# ===========================================================================
ROOT = '/content/drive/MyDrive/Deepfake Detection Project/datasets'

# Set this to YOUR code: GW, AW, NG, NC, SM or PN
MEMBER = 'AW'
assert MEMBER in ['GW','AW','NG','NC','SM','PN'], f"MEMBER must be one of GW, AW, NG, NC, SM, PN. Got: {MEMBER}"

# PILOT = True: tiny slice (10 per class) to test without crashing.
# PILOT = False: real Week 1 pull.
PILOT = True

SEEDS = {'GW': 11, 'AW': 22, 'NG': 33, 'NC': 44, 'SM': 55, 'PN': 66}

if PILOT:
    N_PER_CLASS = {'openfake': 10, 'community': 10, 'reddit': 10}
    MAX_GB, MAX_MINUTES = 0.1, 5
    print('PILOT MODE: 10 per class per source (tiny, RAM-safe)')
else:
    N_PER_CLASS = {'openfake': 1500, 'community': 500, 'reddit': 250}
    MAX_GB, MAX_MINUTES = 2.0, 90
    print('FULL MODE')

print('Drive root:', ROOT)
print('Member:', MEMBER, '| seed:', SEEDS[MEMBER], '| pilot:', PILOT)
print('Per-class targets:', N_PER_CLASS)

## 2. Mount Drive and create folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

FOLDERS = [
    'public_data/openfake', 'public_data/community_forensics',
    'in_the_wild/openfake_reddit', 'in_the_wild/truefake',
    'kampala/originals', 'kampala/whatsapp_photo', 'kampala/whatsapp_document',
    'metadata', 'audit_outputs'
]

def make_folders(root):
    for f in FOLDERS:
        os.makedirs(os.path.join(root, f), exist_ok=True)
    print('Folders ready under:', root)

make_folders(ROOT)

## 3. Source definitions and helpers

RAM-friendly settings:
- Shuffle buffer = 50 (small)
- Metadata flush every 25 images
- PIL images closed explicitly
- One source at a time, with `gc.collect()` between

In [ ]:
EXT = {'JPEG': 'jpg', 'MPO': 'jpg', 'PNG': 'png', 'WEBP': 'webp',
       'GIF': 'gif', 'BMP': 'bmp', 'TIFF': 'tif'}

META_COLS = ['image_id', 'path', 'label', 'source', 'source_detail', 'generator',
             'original_id', 'version', 'role', 'format', 'width', 'height', 'mode',
             'file_bytes', 'sha256', 'license', 'creator', 'pulled_at']

SPECS = {
  'openfake': dict(
      repo='ComplexDataLab/OpenFake', config='core', split='train',
      image_col='image', decode_feature=True, buffer=50,
      label_fn=lambda ex: {'real': 'real', 'fake': 'ai'}.get(str(ex.get('label')).lower()),
      gen_fn=lambda ex, lab: 'real' if lab == 'real' else str(ex.get('model')),
      detail_fn=lambda ex: str(ex.get('type')),
      orig_fn=lambda ex: str(ex.get('row_id')),
      skip_fn=lambda ex: False,
      role='train_val', folder='public_data/openfake', prefix='OF',
      license='CC BY-NC 4.0', cap={'real': 300, 'ai': 40}),
  'community': dict(
      repo='OwensLab/CommunityForensics-Small', config=None, split='train',
      image_col='image_data', decode_feature=False, buffer=100,
      label_fn=lambda ex: {0: 'real', 1: 'ai'}.get(int(ex.get('label'))),
      gen_fn=lambda ex, lab: 'real' if lab == 'real' else str(ex.get('model_name')),
      detail_fn=lambda ex: str(ex.get('real_source') if int(ex.get('label')) == 0
                               else ex.get('architecture')),
      orig_fn=lambda ex: str(ex.get('image_name')),
      skip_fn=lambda ex: bool(ex.get('nsfw_flag')),
      role='train_val', folder='public_data/community_forensics', prefix='CF',
      license='CC BY-NC-SA 4.0', cap={'real': 150, 'ai': 3}),
  'reddit': dict(
      repo='ComplexDataLab/OpenFake', config='reddit', split=None,
      image_col='image', decode_feature=True, buffer=50,
      label_fn=lambda ex: {'real': 'real', 'fake': 'ai'}.get(str(ex.get('label')).lower()),
      gen_fn=lambda ex, lab: 'real' if lab == 'real' else str(ex.get('model')),
      detail_fn=lambda ex: str(ex.get('type')),
      orig_fn=lambda ex: str(ex.get('row_id')),
      skip_fn=lambda ex: False,
      role='wild_test', folder='in_the_wild/openfake_reddit', prefix='OR',
      license='CC BY-NC 4.0', cap={'real': 300, 'ai': 60}),
}

def get_bytes(v):
    if isinstance(v, dict):
        v = v.get('bytes')
    if isinstance(v, (bytes, bytearray)):
        return bytes(v)
    raise ValueError(f'unsupported image value type: {type(v)}')

def group_of(label, generator, detail):
    return generator if label == 'ai' else detail

def meta_path_for(root, name, member):
    return os.path.join(root, 'metadata', f'public_{name}_{member}.csv')

print('SPECS and helpers ready')

## 4. RAM-friendly puller (one source per cell below)

Each source streams from Hugging Face, saves originals to Drive, appends to your member's metadata CSV.
Resumes from existing CSV if the kernel restarts.

In [ ]:
def open_stream(name, seed):
    from datasets import load_dataset, get_dataset_split_names, Image as HFImage
    spec = SPECS[name]
    split = spec['split']
    if split is None:
        splits = get_dataset_split_names(spec['repo'], spec['config'])
        print(f'[{name}] available splits: {splits}')
        split = splits[0]
    ds = load_dataset(spec['repo'], name=spec['config'], split=split, streaming=True)
    if spec['decode_feature']:
        ds = ds.cast_column(spec['image_col'], HFImage(decode=False))
    return ds.shuffle(seed=seed, buffer_size=spec['buffer'])


def pull_from_stream(stream, name, member, root, n_per_class, max_bytes, max_seconds,
                     flush_every=25):
    spec = SPECS[name]
    out_dir = os.path.join(root, spec['folder'])
    os.makedirs(out_dir, exist_ok=True)
    meta_path = meta_path_for(root, name, member)
    os.makedirs(os.path.dirname(meta_path), exist_ok=True)

    counts, per_group, seen, k, used_bytes = Counter(), Counter(), set(), 0, 0

    # Resume from existing CSV (only load needed columns)
    if os.path.exists(meta_path):
        try:
            old = pd.read_csv(meta_path, usecols=['sha256', 'label', 'generator', 'source_detail'])
            for r in old.itertuples(index=False):
                counts[r.label] += 1
                per_group[(r.label, group_of(r.label, r.generator, r.source_detail))] += 1
                seen.add(r.sha256)
            k = len(old)
            print(f'Resuming {name}: {k} images already saved by {member}')
        except Exception as e:
            print(f'Could not resume {name} ({e}); starting fresh')

    buf, skipped, t0, first = [], Counter(), time.time(), True

    def flush():
        if buf:
            need_header = (not os.path.exists(meta_path)) or (os.path.getsize(meta_path) == 0)
            pd.DataFrame(buf, columns=META_COLS).to_csv(
                meta_path, mode='a', header=need_header, index=False)
            buf.clear()

    for ex in stream:
        if time.time() - t0 > max_seconds:
            print(f'[{name}] time limit reached')
            break
        if used_bytes > max_bytes:
            print(f'[{name}] storage limit reached')
            break
        if all(counts[c] >= n_per_class for c in ('real', 'ai')):
            break
        if first:
            print(f'[{name}] columns:', sorted(ex.keys()))
            first = False

        label = spec['label_fn'](ex)
        if label is None or spec['skip_fn'](ex):
            skipped['label_or_flag'] += 1
            continue
        if counts[label] >= n_per_class:
            continue

        gen, detail = spec['gen_fn'](ex, label), spec['detail_fn'](ex)
        g = (label, group_of(label, gen, detail))
        if per_group[g] >= spec['cap'][label]:
            skipped['group_cap'] += 1
            continue

        try:
            raw = get_bytes(ex[spec['image_col']])
            with PILImage.open(io.BytesIO(raw)) as im:
                fmt, (w, h), mode = im.format, im.size, im.mode
        except Exception:
            skipped['unreadable'] += 1
            continue

        sha = hashlib.sha256(raw).hexdigest()
        if sha in seen:
            skipped['duplicate'] += 1
            continue

        k += 1
        image_id = f"{spec['prefix']}_{member}_{k:06d}"
        fname = f"{image_id}.{EXT.get(fmt, (fmt or 'bin').lower())}"
        with open(os.path.join(out_dir, fname), 'wb') as f:
            f.write(raw)

        seen.add(sha)
        counts[label] += 1
        per_group[g] += 1
        used_bytes += len(raw)
        buf.append(dict(image_id=image_id, path=os.path.join(spec['folder'], fname),
                        label=label, source=name, source_detail=detail, generator=gen,
                        original_id=spec['orig_fn'](ex), version='orig', role=spec['role'],
                        format=fmt, width=w, height=h, mode=mode, file_bytes=len(raw),
                        sha256=sha, license=spec['license'], creator=member,
                        pulled_at=time.strftime('%Y-%m-%d %H:%M:%S')))

        if len(buf) >= flush_every:
            flush()
            print(f'[{name}] saved: {dict(counts)}  (skipped: {dict(skipped)})')

    flush()
    gc.collect()
    return dict(source=name, member=member, counts=dict(counts), skipped=dict(skipped),
                gigabytes=round(used_bytes / 1e9, 3), seconds=round(time.time() - t0))

print('Puller functions ready')

## 5a. Pull: OpenFake (source 1 of 3)

Run this cell. If it completes, move to 5b. If the kernel crashes, restart and rerun 5a (it will resume).

In [ ]:
name = 'openfake'
print(f'=== Pulling {name} for member {MEMBER} ===')
stream = open_stream(name, SEEDS[MEMBER])
res = pull_from_stream(stream, name, MEMBER, ROOT, N_PER_CLASS[name],
                       MAX_GB * 1e9, MAX_MINUTES * 60)
print(res)
del stream
gc.collect()
print(f'=== Finished {name} ===')

## 5b. Pull: Community Forensics (source 2 of 3)

Run only after 5a completes successfully.

In [ ]:
name = 'community'
print(f'=== Pulling {name} for member {MEMBER} ===')
stream = open_stream(name, SEEDS[MEMBER])
res = pull_from_stream(stream, name, MEMBER, ROOT, N_PER_CLASS[name],
                       MAX_GB * 1e9, MAX_MINUTES * 60)
print(res)
del stream
gc.collect()
print(f'=== Finished {name} ===')

## 5c. Pull: OpenFake Reddit (source 3 of 3)

Run only after 5b completes successfully.

In [ ]:
name = 'reddit'
print(f'=== Pulling {name} for member {MEMBER} ===')
stream = open_stream(name, SEEDS[MEMBER])
res = pull_from_stream(stream, name, MEMBER, ROOT, N_PER_CLASS[name],
                       MAX_GB * 1e9, MAX_MINUTES * 60)
print(res)
del stream
gc.collect()
print(f'=== Finished {name} ===')

## 6. Sanity check

Lists your member's metadata files and row counts.

In [ ]:
meta_dir = os.path.join(ROOT, 'metadata')
files = sorted(glob.glob(os.path.join(meta_dir, f'public_*_{MEMBER}.csv')))
print(f'Found {len(files)} metadata file(s) for member {MEMBER}')
for f in files:
    try:
        df = pd.read_csv(f)
        print(os.path.basename(f), '| rows:', len(df), '| labels:', dict(df.label.value_counts()) if 'label' in df else '?')
    except Exception as e:
        print(os.path.basename(f), '| ERROR:', e)

if not files:
    print('No metadata yet — run the pull cells above first.')

## What comes next

1. If pilot works (`PILOT = True` completes without crash), set `PILOT = False` and re-run cells 5a–5c.
2. For other members: **restart the runtime**, change `MEMBER`, and repeat.
3. Week 2: run `src/train_baseline.py` (Run A) and `src/train_transfer.py` (Run B) in Colab.

Never run all 6 members in one Colab session — restart between members.